# Tugas Praktikum Big Data Pertemuan 7 — Pipeline Analisis Kedai "Kopi Nusantara"

**Nama:** Sholahuddin Ahmad  
**NPM:** 250506070  

In [5]:
import os
import sys
import getpass
import json
import numpy as np
import pandas as pd

np.random.seed(123)

# Sumber 1: Data cabang (JSON Lines) — 3 cabang operasional
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {
        "cabang_id": i + 1, 
        "nama_cabang": c, 
        "kota": c.split()[-1],
        "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])
    }
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV) — 15 varian produk
menu_list = [
    {
        "menu_id": i, 
        "nama_menu": f"Menu-{i}",
        "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
        "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))
    }
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — 4.000 transaksi pelanggan
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

# Ingestion langsung ke direktori HDFS
username = os.environ.get("USER", "kaky")
os.system(f"hdfs dfs -mkdir -p /user/{username}/tugas7/raw 2>/dev/null")
os.system(f"hdfs dfs -put -f tugas7_transaksi.csv /user/{username}/tugas7/raw/ 2>/dev/null")
os.system(f"hdfs dfs -put -f tugas7_cabang.json /user/{username}/tugas7/raw/ 2>/dev/null")
os.system(f"hdfs dfs -put -f tugas7_menu.csv /user/{username}/tugas7/raw/ 2>/dev/null")

print("Tiga sumber data berhasil diunggah ke klaster HDFS:")
print(f"- Cabang    : {len(data_cabang)} baris")
print(f"- Menu      : {len(menu_list)} baris")
print(f"- Transaksi : {n:,} baris")

Tiga sumber data berhasil diunggah ke klaster HDFS:
- Cabang    : 3 baris
- Menu      : 15 baris
- Transaksi : 4,000 baris


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, avg, round as spark_round, row_number
from pyspark.sql.window import Window

# Isolasi environment native Arch Linux
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder \
    .appName("Tugas7-EndToEnd-SholahuddinAhmad-250506070") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark Engine : Apache Spark {spark.version} | Node: Local Isolated")

Spark Engine : Apache Spark 3.5.9 | Node: Local Isolated


---
### Bagian A

In [7]:
hdfs_raw = f"hdfs://localhost:9000/user/{username}/tugas7/raw"

df_cabang = spark.read.json(f"{hdfs_raw}/tugas7_cabang.json")
df_menu = spark.read.csv(f"{hdfs_raw}/tugas7_menu.csv", header=True, inferSchema=True)
df_transaksi = spark.read.csv(f"{hdfs_raw}/tugas7_transaksi.csv", header=True, inferSchema=True)

print(f"Ekstraksi Selesai:")
print(f"• Cabang    : {df_cabang.count()} data")
print(f"• Menu      : {df_menu.count()} data")
print(f"• Transaksi : {df_transaksi.count():,} data")

Ekstraksi Selesai:
• Cabang    : 3 data
• Menu      : 15 data
• Transaksi : 4,000 data


---
### Bagian B

In [8]:
# Menggabungkan ketiga tabel relasional
df_gabung = df_transaksi \
    .join(df_cabang, on="cabang_id", how="inner") \
    .join(df_menu, on="menu_id", how="inner") \
    .withColumn("total_penjualan", col("qty") * col("harga"))

print(f"Total baris setelah join: {df_gabung.count():,} (Integritas data 100% utuh)")
df_gabung.select("trx_id", "nama_cabang", "kota", "nama_menu", "kategori_menu", "qty", "harga", "total_penjualan").show(5)

Total baris setelah join: 4,000 (Integritas data 100% utuh)
+------+--------------------+----------+---------+--------------+---+-----+---------------+
|trx_id|         nama_cabang|      kota|nama_menu| kategori_menu|qty|harga|total_penjualan|
+------+--------------------+----------+---------+--------------+---+-----+---------------+
|   KN0|Kopi Nusantara Ma...|  Magelang|  Menu-14|          Kopi|  4|18000|          72000|
|   KN1|Kopi Nusantara Ma...|  Magelang|  Menu-11|          Kopi|  4|22000|          88000|
|   KN2|Kopi Nusantara Ma...|  Magelang|  Menu-11|          Kopi|  3|22000|          66000|
|   KN3|Kopi Nusantara Se...|  Semarang|  Menu-11|          Kopi|  1|22000|          22000|
|   KN4|Kopi Nusantara Yo...|Yogyakarta|  Menu-12|Makanan Ringan|  4|35000|         140000|
+------+--------------------+----------+---------+--------------+---+-----+---------------+
only showing top 5 rows



---
### Bagian C

In [10]:
# Agregasi omset per cabang dan menu
omset_menu_cabang = df_gabung.groupBy("nama_cabang", "nama_menu", "kategori_menu").agg(
    spark_sum("total_penjualan").alias("total_omset_menu")
)

# Window partition per cabang terurut omset terbesar
window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_omset_menu").desc())

top2_menu_cabang = omset_menu_cabang \
    .withColumn("peringkat", row_number().over(window_cabang)) \
    .filter(col("peringkat") <= 2)

print("=== TOP-2 MENU TERLARIS DI SETIAP CABANG ===")
top2_menu_cabang.orderBy("nama_cabang", "peringkat").show()

=== TOP-2 MENU TERLARIS DI SETIAP CABANG ===


[Stage 32:>                                                         (0 + 1) / 1]

+--------------------+---------+--------------+----------------+---------+
|         nama_cabang|nama_menu| kategori_menu|total_omset_menu|peringkat|
+--------------------+---------+--------------+----------------+---------+
|Kopi Nusantara Ma...|   Menu-1|Makanan Ringan|         8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|Makanan Ringan|         8085000|        2|
|Kopi Nusantara Se...|  Menu-13|Makanan Ringan|         8750000|        1|
|Kopi Nusantara Se...|   Menu-9|Makanan Ringan|         8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|         9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|      Non-Kopi|         8160000|        2|
+--------------------+---------+--------------+----------------+---------+



---
### Bagian D

In [6]:
# Registrasi Temp View
df_gabung.createOrReplaceTempView("kopi_nusantara")

# Kueri Spark SQL agregasi penjualan per kepala barista
omset_barista = spark.sql("""
    SELECT 
        kepala_barista,
        COUNT(trx_id) AS total_transaksi,
        SUM(total_penjualan) AS total_omset
    FROM kopi_nusantara
    GROUP BY kepala_barista
    ORDER BY total_omset DESC
""")

print("=== KONTRIBUSI PENJUALAN PER KEPALA BARISTA ===")
omset_barista.show()

=== KONTRIBUSI PENJUALAN PER KEPALA BARISTA ===


+--------------+---------------+-----------+
|kepala_barista|total_transaksi|total_omset|
+--------------+---------------+-----------+
|         Nadia|           2695|  177680000|
|          Reza|           1305|   84645000|
+--------------+---------------+-----------+



---
### Bagian E — LOAD (Menyimpan Parquet Terpartisi ke HDFS)

Simpan hasil transformasi ke HDFS dalam format Parquet terpartisi `kategori_menu`, lalu verifikasi dengan membaca kembali berkasnya:

In [7]:
hdfs_output = f"hdfs://localhost:9000/user/{username}/tugas7/processed/laporan"

# Tulis ke HDFS dalam format Parquet terpartisi kategori_menu
df_gabung.write.mode("overwrite").partitionBy("kategori_menu").parquet(hdfs_output)
print(f"Laporan berhasil ditulis ke HDFS: {hdfs_output}")

# Verifikasi pembacaan ulang (harus tepat 4.000 baris)
df_verifikasi = spark.read.parquet(hdfs_output)
print(f"Verifikasi pembacaan ulang: {df_verifikasi.count():,} baris utuh 100%!")

# Periksa direktori fisik partisi di HDFS
!hdfs dfs -ls {hdfs_output}

Laporan berhasil ditulis ke HDFS: hdfs://localhost:9000/user/kaky/tugas7/processed/laporan


Verifikasi pembacaan ulang: 4,000 baris utuh 100%!


Found 4 items
-rw-r--r--   3 kaky supergroup          0 2026-10-09 10:21 hdfs://localhost:9000/user/kaky/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - kaky supergroup          0 2026-10-09 10:21 hdfs://localhost:9000/user/kaky/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - kaky supergroup          0 2026-10-09 10:21 hdfs://localhost:9000/user/kaky/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - kaky supergroup          0 2026-10-09 10:21 hdfs://localhost:9000/user/kaky/tugas7/processed/laporan/kategori_menu=Non-Kopi


---
### Bagian F — Executive Summary & Rekomendasi Strategis Bisnis

Langkah pamungkas: buat tabel performa cabang lalu rumuskan rekomendasi bisnis untuk jajaran manajemen Kopi Nusantara:

In [8]:
ringkasan_cabang = df_gabung.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    spark_round(avg("total_penjualan"), 2).alias("rata_rata_transaksi")
).orderBy(col("total_penjualan").desc())

print("=== RINGKASAN KINERJA EKSEKUTIF KOPI NUSANTARA ===")
ringkasan_cabang.show()

=== RINGKASAN KINERJA EKSEKUTIF KOPI NUSANTARA ===


+--------------------+----------+--------------+---------------+----------------+-------------------+
|         nama_cabang|      kota|kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_transaksi|
+--------------------+----------+--------------+---------------+----------------+-------------------+
|Kopi Nusantara Se...|  Semarang|         Nadia|       90542000|            1393|           64997.85|
|Kopi Nusantara Ma...|  Magelang|         Nadia|       87138000|            1302|           66926.27|
|Kopi Nusantara Yo...|Yogyakarta|          Reza|       84645000|            1305|           64862.07|
+--------------------+----------+--------------+---------------+----------------+-------------------+



#### Analisis Bisnis & Rekomendasi Manajerial (Kopi Nusantara)

Berdasarkan hasil analisis data triwulan, cabang **Kopi Nusantara Semarang** mencatatkan performa terbaik dengan torehan omset tertinggi mencapai **Rp 90.542.000** dan volume transaksi terbanyak sebanyak **1.393 transaksi**. Sebaliknya, cabang **Kopi Nusantara Yogyakarta** menjadi cabang dengan kinerja terlemah dengan total perolehan omset **Rp 84.645.000** serta nilai rata-rata per transaksi paling rendah (**Rp 64.862,07 per transaksi**).

Temuan paling menarik dari analisis *Window Function* adalah dominasi kategori **Makanan Ringan** di semua cabang, di mana menu terlaris Yogyakarta dipegang oleh Menu-12 (Rp 9.275.000) dan Semarang oleh Menu-13 (Rp 8.750.000). Produk berbasis kopi justru mengalami stagnasi penjualan di Yogyakarta dan kalah bersaing melawan minuman non-kopi (Menu-15).

Untuk mengangkat performa cabang Yogyakarta, manajemen disarankan meluncurkan strategi promosi **Bundling Promo (Coffee Pairing)**, yaitu memaketkan menu kopi unggulan bersama Menu-12 dengan potongan diskon 15% pada jam sibuk (*afternoon break*). Selain itu, pelatihan standar racikan dan pelayanan barista Reza perlu diselaraskan dengan SOP barista Nadia yang terbukti sukses memimpin performa omset di dua cabang teratas.

---
## Menutup Sesi PySpark

Menghentikan runtime PySpark agar alokasi resource CPU dan RAM dikembalikan ke sistem host:

In [9]:
# Menghentikan runtime PySpark
spark.stop()
print("SparkSession berhasil dihentikan. Alokasi resource CPU dan RAM telah dikembalikan ke sistem host.")

SparkSession berhasil dihentikan. Alokasi resource CPU dan RAM telah dikembalikan ke sistem host.
